## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Airflow`.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    col,
    lit
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("lab 1 Test")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user0/.ivy2/cache
The jars for the packages stored in: /home/user0/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8ea27889-6fd3-4290-9ebd-87ae171e61fe;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 627ms :: artifacts dl 22ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

Для исследования будем использовать датасет `"US Used cars dataset"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/ananaymital/us-used-cars-dataset.

Датасет включает в себя информацию о более чем трех миллионах используемых машин в США. Он разрешен для использования в учебных целях.

Данный датасет уже загружен в `HDFS` по адресу: `hdfs:///datasets/used_cars_data.csv`

Указываем путь в `HDFS` для файла с данными.

In [5]:
#path = "hdfs:///datasets/used_cars_data.csv"
path = "hdfs:///user/user0/datasets/CICIoT2023.csv"

Заполняем датафрейм данными из файла.

In [6]:
df = (spark.read.format("csv")
      .option("header", "true")
      .load(path)
)

Выводим фрагмент датафрейма на экран.

In [7]:
df.show()

26/11/08 21:29:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+-------------+-------------+--------+------------------+------------------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+---------+---------+---------+----+-----+---+------+----+---+---+---+---+----+---+----+---+---+-------+-----+------+-----------------+------------------+--------+-----------------+------+------------------+------------------+------------------+--------+------+--------------------+
|     flow_duration|Header_Length|Protocol Type|Duration|              Rate|             Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|fin_count|urg_count|rst_count|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IPv|LLC|Tot sum|  Min|   Max|              AVG|               Std|Tot size|              IAT|Number|          Magnitue|            Radius|        Covariance|Variance

Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все солбцы датасета. Оставим следующие колонки, удалив остальные:
<!--

| Название столбца  | Расшифровка |
| ------------- | ------------- |
| vin               | Идентификационный номер автомобиля  |
| body_type         | Тип кузова автомобиля (кабриолет, хэтчбек, седан и т.д.)  |
| daysonmarket      | Количество дней, прошедших с момента первого размещения автомобиля на сайте |
| fleet             | Был ли автомобиль ранее частью автопарка |
| has_accidents     | Имеется ли регистрация аварий на данном автомобиле |
| horsepower        | Мощность двигателя в лошадиных силах |
| is_certified      | Является ли автомобиль сертифицированным (на сертифицированные автомобили распространяется гарантийный срок) |
| is_cpo            | Подержанные автомобили, сертифицированные дилером |
| is_oemcpo         | Подержанные автомобили, сертифицированные производителем |
| major_options     | Список основный опций автомобиля |
| maximum_seating   | Максимальное количество посадочных мест |
| mileage           | Величина пробега автомобиля |
| price             | Цена автомобиля |
| wheel_system      | Тип привода |
| year              | Год выпуска автомобиля |
-->


| Название столбца | Расшифровка |
| ---------------- | ----------- |
| flow_duration | Длительность потока |
| Header_Length | Длина заголовка |
| Protocol Type | Тип протокола |
| Duration | Time_to_live |
| Rate | Скорость передачи пакета в потоке |
| Srate | Скорость передачи исходящих пакетов в потоке |
| Drate | Скорость передачи входящих пакетов в потоке |
| fin_flag_number | Значение флага FIN |
| syn_flag_number | Значение флага SYN |
| rst_flag_number | Значение флага RST |
| psh_flag_number | Значение флага PSH |
| ack_flag_number | Значение флага ACK |
| ece_flag_number | Значение флага ECE |
| cwr_flag_number | Значение флага CWR |
| ack_count | Кол-во пакетов с установленным флагом ACK в одном потоке |
| syn_count | Кол-во пакетов с установленным флагом SYN в одном потоке|
| fin_count | Кол-во пакетов с установленным флагом FIN в одном потоке|
| urg_count | Кол-во пакетов с установленным флагом URG в одном потоке|
| rst_count | Кол-во пакетов с установленным флагом RST в одном потоке|
| HTTP | Признак трафика HTTP |
| HTTPS | Признак трафика HTTPS |
| DNS | Признак трафика DNS |
| Telnet | Признак трафика Telnet |
| SMTP | Признак трафика SMTP |
| SSH | Признак трафика SSH |
| IRC | Признак трафика IRC |
| TCP | Признак трафика TCP |
| UDP | Признак трафика UDP |
| DHCP | Признак трафика DHCP |
| ARP | Признак трафика ARP |
| ICMP | Признак трафика ICMP |
| IPv | Признак трафика IPv |
| LLC | Признак трафика LLC |
| Tot sum | Общая сумма длинны пакетов в потоке |
| Min | Минимальная длина пакета в потоке |
| Max | Максимальныя длина пакета в потоке |
| AVG | Среднее значение длины пакета в потоке |
| Std | Стандартное отклонение длинны пакета в потоке |
| Tot size | Длинна пакета |
| IAT | Межпакетный интервал |
| Number | Количество пакетов в потоке |
| Magnitue | Модуль/амплитуда (сред знач длин входящих пакетов в потоке + сред знач длин изходящих пакетов в потоке )**0,5 |
| Radius | Радиус разброса значений (разница знач длин входящих пакетов в потоке + разница знач длин изходящих пакетов в потоке)**0,5 |
| Covariance | Ковариация длинны исхлдящих и входящих пакетов |
| Variance | Разница длинн входящих пакетов / разница длинн исходящих пакетов |
| Weight | Кол-во входящих пакетов * кол-во исходящих пакетов |
| label | Тип трафика / атаки |

In [8]:
"""
flow_duration|Header_Length|Protocol Type|Duration|              Rate|             Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|fin_count|urg_count|rst_count|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IPv|LLC|Tot sum|  Min|   Max|              AVG|               Std|Tot size|              IAT|Number|          Magnitue|            Radius|        Covariance|Variance|Weight|               label
"""
df = df.select("flow_duration", "Header_Length", "Duration", "Rate", "Srate", "Drate", "fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number", "ack_flag_number", "ece_flag_number", "cwr_flag_number", "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "Tot sum", "Min", "Max",  "AVG", "Std", "Tot size", "IAT", "Number", "Magnitue", "Radius", "Covariance", "Variance", "Weight", "label")

In [9]:
df.show()

+------------------+-------------+--------+------------------+------------------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----+-----+---+------+----+---+---+---+---+----+---+----+---+-------+-----+------+-----------------+------------------+--------+-----------------+------+------------------+------------------+------------------+--------+------+--------------------+
|     flow_duration|Header_Length|Duration|              Rate|             Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IPv|Tot sum|  Min|   Max|              AVG|               Std|Tot size|              IAT|Number|          Magnitue|            Radius|        Covariance|Variance|Weight|               label|
+------------------+-------------+--------+------------------+------------------+-----+---------------+---

Выведем на экран метаданные датасета.

In [10]:
df.printSchema()

root
 |-- flow_duration: string (nullable = true)
 |-- Header_Length: string (nullable = true)
 |-- Duration: string (nullable = true)
 |-- Rate: string (nullable = true)
 |-- Srate: string (nullable = true)
 |-- Drate: string (nullable = true)
 |-- fin_flag_number: string (nullable = true)
 |-- syn_flag_number: string (nullable = true)
 |-- rst_flag_number: string (nullable = true)
 |-- psh_flag_number: string (nullable = true)
 |-- ack_flag_number: string (nullable = true)
 |-- ece_flag_number: string (nullable = true)
 |-- cwr_flag_number: string (nullable = true)
 |-- HTTP: string (nullable = true)
 |-- HTTPS: string (nullable = true)
 |-- DNS: string (nullable = true)
 |-- Telnet: string (nullable = true)
 |-- SMTP: string (nullable = true)
 |-- SSH: string (nullable = true)
 |-- IRC: string (nullable = true)
 |-- TCP: string (nullable = true)
 |-- UDP: string (nullable = true)
 |-- DHCP: string (nullable = true)
 |-- ARP: string (nullable = true)
 |-- ICMP: string (nullable = tru

Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

In [11]:
"""
 flow_duration|Header_Length|Duration|              Rate|             Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IPv|Tot sum|  Min|   Max|              AVG|               Std|Tot size|              IAT|Number|          Magnitue|            Radius|        Covariance|Variance|Weight|               label     
"""
def transform_dataframe(data: DataFrame) -> DataFrame:
    double_cols = [
    "flow_duration", "Header_Length", "Duration", "Rate", "Srate", "Drate",
    "Tot sum", "Min", "Max",  "AVG", "Std", "Tot size", "IAT", "Number", 
    "Magnitue", "Radius", "Covariance", "Variance", "Weight"
    ]

    int_col = [
        "fin_flag_number", "syn_flag_number", "rst_flag_number", 
        "psh_flag_number", "ack_flag_number", "ece_flag_number", 
        "cwr_flag_number", "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", 
        "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv"
    ]

    #строковые типы данных меняем на double

    for c in double_cols:
        data = data.withColumn(c, col(c).cast("double"))

    for c in int_col:
        data = data.withColumn(c, col(c).cast("int"))

    


    return data


In [12]:
df = transform_dataframe(df)

In [13]:
df.show()

+------------------+-------------+--------+------------------+------------------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----+-----+---+------+----+---+---+---+---+----+---+----+---+-------+-----+------+-----------------+------------------+--------+-------------------+------+------------------+------------------+------------------+--------+------+--------------------+
|     flow_duration|Header_Length|Duration|              Rate|             Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IPv|Tot sum|  Min|   Max|              AVG|               Std|Tot size|                IAT|Number|          Magnitue|            Radius|        Covariance|Variance|Weight|               label|
+------------------+-------------+--------+------------------+------------------+-----+---------------

In [14]:
df.printSchema()

root
 |-- flow_duration: double (nullable = true)
 |-- Header_Length: double (nullable = true)
 |-- Duration: double (nullable = true)
 |-- Rate: double (nullable = true)
 |-- Srate: double (nullable = true)
 |-- Drate: double (nullable = true)
 |-- fin_flag_number: integer (nullable = true)
 |-- syn_flag_number: integer (nullable = true)
 |-- rst_flag_number: integer (nullable = true)
 |-- psh_flag_number: integer (nullable = true)
 |-- ack_flag_number: integer (nullable = true)
 |-- ece_flag_number: integer (nullable = true)
 |-- cwr_flag_number: integer (nullable = true)
 |-- HTTP: integer (nullable = true)
 |-- HTTPS: integer (nullable = true)
 |-- DNS: integer (nullable = true)
 |-- Telnet: integer (nullable = true)
 |-- SMTP: integer (nullable = true)
 |-- SSH: integer (nullable = true)
 |-- IRC: integer (nullable = true)
 |-- TCP: integer (nullable = true)
 |-- UDP: integer (nullable = true)
 |-- DHCP: integer (nullable = true)
 |-- ARP: integer (nullable = true)
 |-- ICMP: inte

Видно, что теперь столбцы датафрейма содержат значения корректных типов.

Полученный датафрейм сохраним для дальнейшего использования. Сохранение выполним в таблицу `Apache Iceberg`. 

`Apache Iceberg` — это поддерживающий высокую производительность табличный формат для больших данных.

Сначала создадим базу данных, в которой будет расположена таблица. Во избежание путаницы, **каждую таблицу следует называть с использованием фамилии студента**.

In [15]:
database_name = "dolgova_database"

Создадим инструкцию SQL для добавления базы данных в каталог `Apache Spark`.

In [16]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}
"""

In [17]:
spark.sql(create_database_sql)

DataFrame[]

Установим созданную базу данных как текущую.

In [18]:
spark.catalog.setCurrentDatabase(database_name)

И, наконец, записываем преобразованный датафрейм в таблицу `sobd_lab1_table`.

In [21]:
# Сохранение DataFrame в виде таблицы
df.writeTo("sobd_lab1_table").using("iceberg").create()

После успешной записи можно посмотреть, какие таблицы входят в базу данных.

In [22]:
for table in spark.catalog.listTables():
    print(table.name)

sobd_lab1_table


Обратите внимание, что при необходимости созданные базу данных и таблицу можно удалить следующими командами.

In [ ]:
#spark.sql("DROP TABLE spark_catalog.dolgova_database.sobd_lab1_table")
# spark.sql("DROP DATABASE spark_catalog.ivanov_database")

DataFrame[]

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [23]:
spark.stop()